In [1]:
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

pd.set_option('display.float_format', lambda value: f'{value:,.3f}')


In [6]:
candidate_paths = [
    Path('../data/Forbes2000.csv'),
    Path('data/Forbes2000.csv'),
    Path('Forbes2000.csv'),
]

for csv_path in candidate_paths:
    if csv_path.exists():
        break
else:
    raise FileNotFoundError('Could not locate Forbes2000.csv')

df = pd.read_csv(csv_path, encoding='ISO-8859-1', index_col=0)
features = ['sales', 'profits', 'assets']
model_df = df[['name', 'country', 'category', *features]].dropna(subset=features).copy()

print(f'Loaded {len(df)} companies from {csv_path}.')
print('Missing values in clustering features:')
print(df[features].isna().sum())
print(f'Rows retained for clustering: {len(model_df)}')

model_df[features].describe().round(3)


Loaded 2000 companies from ../data/Forbes2000.csv.
Missing values in clustering features:
sales      0
profits    5
assets     0
dtype: int64
Rows retained for clustering: 1995


,sales,profits,assets
count,"1,995.000","1,995.000","1,995.000"
mean,9.709,0.381,34.068
std,18.024,1.765,99.797
min,0.010,-25.830,0.270
25%,2.010,0.080,4.020
50%,4.360,0.200,9.330
75%,9.575,0.440,22.745
max,256.330,20.960,"1,264.030"


In [7]:
# Standardize the features before KMeans because sales, profits, and assets
# are measured on very different scales.
scaler = StandardScaler()
X = scaler.fit_transform(model_df[features])

models = {}
comparison_rows = []

for k in (2, 3):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    models[k] = kmeans
    comparison_rows.append({
        'k': k,
        'silhouette_score': silhouette_score(X, labels),
        'inertia': kmeans.inertia_,
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('k').round(4)
comparison_df


,silhouette_score,inertia
k,,
2,0.841,"3,779.761"
3,0.837,"3,022.667"


In [8]:
best_k = int(comparison_df['silhouette_score'].idxmax())
best_model = models[best_k]
model_df['cluster'] = best_model.labels_

other_k = 3 if best_k == 2 else 2
print(
    f'Better clustering model: k={best_k} '
    f'(silhouette={comparison_df.loc[best_k, "silhouette_score"]:.4f}; '
    f'k={other_k} has {comparison_df.loc[other_k, "silhouette_score"]:.4f}).'
)

cluster_summary = (
    model_df.groupby('cluster')
    .agg(
        company_count=('cluster', 'size'),
        sales_mean=('sales', 'mean'),
        sales_std=('sales', 'std'),
        profits_mean=('profits', 'mean'),
        profits_std=('profits', 'std'),
        assets_mean=('assets', 'mean'),
        assets_std=('assets', 'std'),
    )
    .round(3)
)

cluster_summary


Better clustering model: k=2 (silhouette=0.8414; k=3 has 0.8371).


,company_count,sales_mean,sales_std,profits_mean,profits_std,assets_mean,assets_std
cluster,,,,,,,
0,81,67.220,50.322,3.867,5.049,380.146,305.033
1,1914,7.275,9.313,0.234,1.283,19.422,34.702
